# Trinity 2035 — AC Redispatch Congestion & Reinforcement Study

Answers one question: **on the un-reinforced 2024 Spanish grid, is the AC OPF redispatch feasible
for the Trinity 2035 EMPIRE portfolio — and if not, exactly which lines need how much more capacity?**

Reuses the method in `docs/method_grid_reinforcement_identification.md` (already applied once to
NECPEssentials — see `results_plots_2035.ipynb` Part II). This notebook is dedicated to Trinity's
congestion-relief study specifically, so the before/after story stays self-contained for the report.

**Three states compared, all `[scenario].label = "Trinity"`, `[redispatch].power_flow = "AC"`:**

| state | `line_rating_factor` | horizon | result |
|---|---|---|---|
| **baseline** ("before") | 0.80 (2024 nameplate, operating derate) | 2 fixed days (8 Jul, 2 Dec 2024) | **0 / 48 hours solved** — every hour either `LOCALLY_INFEASIBLE` or `ITERATION_LIMIT` |
| **congested** | 1.00 (nameplate) | 2 sampled weeks (336 h, cluster run) | 336 / 336 solved, but 17 branches still sit at/near their cap |
| **near-reinforced** ("after", probe) | 1.50 | same 2 sampled weeks (336 h, cluster run) | 336 / 336 solved, only 2 branches still capped |

The baseline never produced a solution, so it has **no branch-flow data to plot** — Part 1 shows the
`0/48` fact directly and uses the **congested (LRF=1.00)** run as the closest available feasible
state to visualise *where* the strain concentrates. Part 2 sizes the fix from the **near-reinforced
(LRF=1.50)** run, which is far less saturated. Two branches are still pinned even at 1.50× — their
true requirement is only bounded (`≥1.50×`), not pinned exactly; closing that gap needs one more,
longer cluster probe that was judged not worth the runtime for this report.


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pio.templates.default = 'plotly_white'

# ---- where each of the three states lives on disk ----
BASELINE_DIR   = Path('results/Trinity_ac_lrf_study/baseline_lrf0.80')   # LRF=0.80, AC, 2 days -> 0/48 solved
CONGESTED_DIR  = Path('results/Trinity/cluster/line_rating_factor_1')    # LRF=1.00, AC, 2 weeks -> 336/336
REINFORCED_DIR = Path('results/Trinity/cluster/line_rating_factor_1.5')  # LRF=1.50, AC, 2 weeks -> 336/336

LRF_CONGESTED  = 1.00
LRF_REINFORCED = 1.50

OUT = Path('results/Trinity/reinforcement'); OUT.mkdir(parents=True, exist_ok=True)

def save_plotly(fig, filename):
    out = OUT / filename
    fig.write_html(out, include_plotlyjs='cdn', full_html=True)
    return out

# Spain geo frame reused by every map (same as results_plots_2035.ipynb)
GEO = dict(projection_type='mercator', lonaxis_range=[-10, 5], lataxis_range=[35.5, 44.5],
           showland=True, landcolor='#f5f5f2', showcoastlines=True, coastlinecolor='#888888',
           showcountries=True, countrycolor='#aaaaaa', showocean=True, oceancolor='#eef5fb')

print('output folder:', OUT.resolve())


## Setup — load the three states + the static network

In [ ]:
# ---- baseline: only a summary exists (every hour failed, nothing to plot) ----
base_summary = pd.read_csv(BASELINE_DIR / 'summary.csv')

# ---- congested (LRF=1.00) and near-reinforced (LRF=1.50): full redispatch output ----
cong_summary = pd.read_csv(CONGESTED_DIR / 'summary.csv')
cong_branch  = pd.read_csv(CONGESTED_DIR / 'branch_flows.csv')
cong_volt    = pd.read_csv(CONGESTED_DIR / 'bus_voltages.csv')

rein_summary = pd.read_csv(REINFORCED_DIR / 'summary.csv')
rein_branch  = pd.read_csv(REINFORCED_DIR / 'branch_flows.csv')
rein_volt    = pd.read_csv(REINFORCED_DIR / 'bus_voltages.csv')

# ---- static network: bus coordinates (x=lon, y=lat) and nominal line ratings ----
# Bus_Data.csv carries a UTF-8 BOM on the first header -> utf-8-sig.
buses = pd.read_csv('Data/Bus_Data.csv', encoding='utf-8-sig')
buses_pos = buses.reset_index(drop=True).copy()
buses_pos.index = range(1, len(buses_pos) + 1)   # branch from_bus/to_bus are 1-based row indices

lines_g = pd.read_csv('Data/lines.csv')

def solved_count(df):
    return int(df['status'].isin(['OPTIMAL', 'LOCALLY_SOLVED']).sum()), len(df)

b_ok, b_n = solved_count(base_summary)
c_ok, c_n = solved_count(cong_summary)
r_ok, r_n = solved_count(rein_summary)
print(f'baseline   (LRF=0.80, 2 days) : {b_ok}/{b_n} solved')
print(f'congested  (LRF=1.00, 2 wks)  : {c_ok}/{c_n} solved')
print(f'reinforced (LRF=1.50, 2 wks)  : {r_ok}/{r_n} solved')


## Part 1 — Before: is the current (2024) grid adequate?

At the operating derate the rest of the market chain uses (`line_rating_factor = 0.80`), the AC OPF
redispatch is **completely infeasible on the Trinity 2035 portfolio** — every one of 48 tested hours
fails (voltage relaxation alone does not fix it, see the project log; the mechanism is thermal, the
same conclusion reached earlier for NECPEssentials).

Because no hour actually solved at 0.80, there is no flow solution to draw. The map below instead
uses the **congested (LRF = 1.00) state** — nameplate rating, no operating derate — as the closest
available feasible reference: it already shows a clear stress pattern concentrated on a handful of
corridors, which is exactly what has to give at the tighter 0.80 operating limit.

In [ ]:
fail_counts = base_summary['status'].value_counts()
print('baseline (LRF=0.80) failure breakdown, 48 hours:')
print(fail_counts.to_string())


### Where the strain concentrates (congested state, LRF = 1.00)

In [ ]:
# peak loading (%) of nameplate reached by each branch over the 336-hour sample
peak_cong = (cong_branch.groupby(['branch_id', 'branch_name', 'from_bus', 'to_bus'])['loading_pct']
             .max().reset_index().rename(columns={'loading_pct': 'peak_pct'}))

bins   = [0, 60, 85, 95, 100.001]
colors = ['#c9ccd1', '#fee08b', '#fc8d59', '#d73027']
widths = [0.6, 1.4, 2.2, 3.2]
labels = ['< 60 %', '60-85 %', '85-95 %', '\u2265 95 % (stressed)']

fig = go.Figure()
for (lo, hi, col, w, lbl) in zip(bins[:-1], bins[1:], colors, widths, labels):
    sel = peak_cong[(peak_cong['peak_pct'] >= lo) & (peak_cong['peak_pct'] < hi)]
    if sel.empty:
        continue
    lons, lats = [], []
    for _, r in sel.iterrows():
        b0, b1 = buses_pos.loc[int(r['from_bus'])], buses_pos.loc[int(r['to_bus'])]
        lons += [b0['x'], b1['x'], None]; lats += [b0['y'], b1['y'], None]
    fig.add_trace(go.Scattergeo(lon=lons, lat=lats, mode='lines',
                                line=dict(width=w, color=col), name=lbl, hoverinfo='skip'))

stressed = peak_cong[peak_cong['peak_pct'] >= 95].sort_values('peak_pct', ascending=False)
tlon = [(buses_pos.loc[int(r['from_bus'])]['x'] + buses_pos.loc[int(r['to_bus'])]['x']) / 2
        for _, r in stressed.iterrows()]
tlat = [(buses_pos.loc[int(r['from_bus'])]['y'] + buses_pos.loc[int(r['to_bus'])]['y']) / 2
        for _, r in stressed.iterrows()]
fig.add_trace(go.Scattergeo(lon=tlon, lat=tlat, mode='markers+text',
    text=[f"{n}<br>{p:.0f}%" for n, p in zip(stressed['branch_name'], stressed['peak_pct'])],
    textposition='top center', textfont=dict(size=9, color='#7a0177'),
    marker=dict(size=6, color='#7a0177'), name='stressed (\u226595%)'))

fig.update_geos(**GEO)
fig.update_layout(title='Trinity 2035 \u2014 peak branch loading at nameplate rating (LRF=1.00, 2-week sample)',
                  height=680, margin=dict(l=10, r=10, t=60, b=10), legend=dict(y=0.9))
save_plotly(fig, 'stress_map_lrf1.00.html'); fig.show()

print(f"{len(stressed)} branches reach >=95% of nameplate at LRF=1.00, out of {len(peak_cong)} total")


## Part 2 — Sizing the reinforcement

For every branch, the minimum rating factor it needs is
`req_factor = peak_loading_pct \u00d7 LRF_used / 100`, read off the **near-reinforced (LRF=1.50)**
run — far less saturated than the LRF=1.00 run, so most branches' true peak is fully visible here.

Two branches are still pinned at (or essentially at) their 1.50\u00d7 cap — their `req_factor` below
is a **lower bound**, not their true requirement.

In [ ]:
req = (rein_branch.groupby(['branch_id', 'branch_name', 'from_bus', 'to_bus'])['loading_pct']
       .max().reset_index().rename(columns={'loading_pct': 'peak_pct'}))
req['req_factor'] = req['peak_pct'] * LRF_REINFORCED / 100.0
req['capped']     = req['peak_pct'] >= 99.5   # still pinned at the LRF=1.50 ceiling -> lower bound only
req = req.sort_values('req_factor', ascending=False)

thresholds = [0.8, 1.0, 1.5, 2.0, 3.0]
counts = [int((req['req_factor'] > t).sum()) for t in thresholds]
print(f'{len(req)} branches total; system-min feasible line_rating_factor >= %.2f (branch %s)'
      % (req['req_factor'].max(), req.iloc[0]['branch_name']))
for t, c in zip(thresholds, counts):
    print(f'  branches needing > {t:>3}x : {c}')
n_capped = int(req['capped'].sum())
print(f'  of which still capped at {LRF_REINFORCED:g}x (true need unresolved, lower bound only): {n_capped}')

need = req[req['req_factor'] > 1.0].copy()
fig = make_subplots(rows=1, cols=2, column_widths=[0.42, 0.58],
                    subplot_titles=('branches above a rating factor', f'the {len(need)} branches needing reinforcement'),
                    specs=[[{'type': 'bar'}, {'type': 'bar'}]])
fig.add_bar(x=[f'>{t}\u00d7' for t in thresholds], y=counts, marker_color='#3987e5',
            text=counts, textposition='outside', row=1, col=1)
top = need.sort_values('req_factor').copy()
bar_colors = ['#7a0177' if c else '#d73027' for c in top['capped']]
bar_suffixes = ['+' if c else '' for c in top['capped']]
fig.add_bar(x=top['req_factor'], y=top['branch_name'], orientation='h',
            marker_color=bar_colors,
            text=[f'{v:.2f}\u00d7{s}' for v, s in zip(top['req_factor'], bar_suffixes)],
            textposition='outside', row=1, col=2)
fig.update_xaxes(title_text='min rating factor needed (purple = lower bound only)', row=1, col=2)
fig.update_yaxes(title_text='branches', row=1, col=1)
fig.update_layout(title='Trinity 2035 \u2014 intra-Spain reinforcement requirement (from LRF=1.50 probe)',
                  height=max(420, 40 * len(need) + 160), showlegend=False)
save_plotly(fig, 'reinforcement_bars_trinity.html'); fig.show()
need


### Reinforcement map — which corridors must grow, and by how much

Grey = adequate at nameplate. Warming colours = needs reinforcement. Branches outlined in **purple**
text are the two still pinned at the LRF=1.50 probe ceiling — treat their number as *at least* that
much, not exact; a practical design margin (e.g. round up to 2\u00d7) is a defensible substitute for
chasing an exact figure that would need a much longer cluster run to pin down.

In [ ]:
import json, urllib.request

bins   = [0, 1.0, 1.5, 2.0, 3.0]
colors = ['#c9ccd1', '#fee08b', '#fc8d59', '#d73027']
widths = [0.6, 1.6, 2.4, 3.4]

# ── NUTS3 borders from Eurostat GISCO (same source as results_plots_2035.ipynb) ──
_nuts_url = ('https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/'
             'NUTS_RG_20M_2021_4326_LEVL_3.geojson')
with urllib.request.urlopen(_nuts_url) as _r:
    _nuts = json.load(_r)
_es_feats = [f for f in _nuts['features'] if f['properties']['CNTR_CODE'] == 'ES']

fig = go.Figure()
for _feat in _es_feats:
    _geom = _feat['geometry']
    _polys = _geom['coordinates'] if _geom['type'] == 'MultiPolygon' else [_geom['coordinates']]
    for _poly in _polys:
        for _ring in _poly:
            _lons = [c[0] for c in _ring] + [_ring[0][0]]
            _lats = [c[1] for c in _ring] + [_ring[0][1]]
            fig.add_trace(go.Scattergeo(lon=_lons, lat=_lats, mode='lines',
                                        line=dict(color='#888888', width=0.8),
                                        showlegend=False, hoverinfo='skip'))

for lo, hi, col, w in zip(bins[:-1], bins[1:], colors, widths):
    sel = req[(req['req_factor'] >= lo) & (req['req_factor'] < hi)]
    if sel.empty:
        continue
    lons, lats = [], []
    for _, r in sel.iterrows():
        b0, b1 = buses_pos.loc[int(r['from_bus'])], buses_pos.loc[int(r['to_bus'])]
        lons += [b0['x'], b1['x'], None]; lats += [b0['y'], b1['y'], None]
    lbl = 'adequate (\u22641\u00d7)' if hi == 1.0 else f'{lo:g}\u2013{hi:g}\u00d7 needed'
    fig.add_trace(go.Scattergeo(lon=lons, lat=lats, mode='lines',
                                line=dict(width=w, color=col), name=lbl, hoverinfo='skip'))

top5 = req.sort_values('req_factor', ascending=False).head(len(need))
tlon = [(buses_pos.loc[int(r['from_bus'])]['x'] + buses_pos.loc[int(r['to_bus'])]['x']) / 2 for _, r in top5.iterrows()]
tlat = [(buses_pos.loc[int(r['from_bus'])]['y'] + buses_pos.loc[int(r['to_bus'])]['y']) / 2 for _, r in top5.iterrows()]
fig.add_trace(go.Scattergeo(lon=tlon, lat=tlat, mode='markers+text',
    text=[f"{n}<br>{f:.2f}\u00d7{'+' if c else ''}" for n, f, c in zip(top5['branch_name'], top5['req_factor'], top5['capped'])],
    textposition='top center',
    textfont=[dict(size=10, color='#7a0177') if c else dict(size=10, color='#7a0177') for c in top5['capped']],
    marker=dict(size=6, color='#7a0177'), name='needs reinforcement'))
fig.update_geos(**GEO)
fig.update_layout(title='Trinity 2035 \u2014 intra-Spain reinforcement map (min rating factor needed)',
                  height=680, margin=dict(l=10, r=10, t=60, b=10), legend=dict(y=0.9))
save_plotly(fig, 'reinforcement_map_trinity.html'); fig.show()


## Part 3 — After: validated system state at the near-reinforced probe (LRF = 1.50)

This is not yet the exact targeted fix (only these 17 branches reinforced, the rest left at 0.80) —
it is a **blanket** 1.50\u00d7 run, which is a strict superset of the targeted fix and therefore an
upper bound on what reinforcement is needed. It is fully validated: 336/336 hours solved.

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    f'branch loading distribution \u2014 congested (LRF={LRF_CONGESTED:g})',
    f'branch loading distribution \u2014 near-reinforced (LRF={LRF_REINFORCED:g})'))
fig.add_histogram(x=cong_branch['loading_pct'], nbinsx=40, marker_color='#d73027', name='LRF=1.00', row=1, col=1)
fig.add_histogram(x=rein_branch['loading_pct'], nbinsx=40, marker_color='#2e7d32', name='LRF=1.50', row=1, col=2)
fig.update_xaxes(title_text='loading (% of that run\u2019s rating)')
fig.update_yaxes(title_text='branch-hours (log)', type='log')
fig.update_layout(height=420, showlegend=False,
                  title='Congestion relief: branch-hour loading distribution, before vs. after')
save_plotly(fig, 'loading_distribution_before_after.html'); fig.show()

cong_stress_hours = cong_branch[cong_branch['loading_pct'] >= 99.5].groupby(['date', 'hour'])['branch_id'].count()
rein_stress_hours = rein_branch[rein_branch['loading_pct'] >= 99.5].groupby(['date', 'hour'])['branch_id'].count()
print(f'branch-hours >=99.5% loaded : congested={len(cong_branch[cong_branch.loading_pct>=99.5])}  '
      f'near-reinforced={len(rein_branch[rein_branch.loading_pct>=99.5])}  (out of {len(cong_branch)} branch-hours each)')
print(f'distinct hours with >=1 branch >=99.5% loaded : congested={cong_stress_hours.shape[0]}  '
      f'near-reinforced={rein_stress_hours.shape[0]}  (out of 336 hours)')


In [ ]:
# voltage-band compliance (the AC solution really did solve; a DC redispatch would report a flat 1.0 pu)
for label, volt in (('congested (LRF=1.00)', cong_volt), ('near-reinforced (LRF=1.50)', rein_volt)):
    vmin, vmax = volt['vm_pu'].min(), volt['vm_pu'].max()
    out_of_band = ((volt['vm_pu'] < 0.95) | (volt['vm_pu'] > 1.05)).sum()
    flat = np.isclose(volt['vm_pu'], 1.0).all()
    note = '  (FLAT 1.0 -> this was a DC solve, not AC!)' if flat else ''
    print(f'{label:28s}: vm_pu in [{vmin:.4f}, {vmax:.4f}]  |  {out_of_band} bus-hours outside \u00b15%{note}')


## Summary table (for the report)

Nominal ratings recovered from `Data/lines.csv` / transformer data via the actual peak MW flow
(active power) observed at LRF=1.50, cross-checked against the `loading_pct`-based (apparent power)
`req_factor` above. Two rows are marked **lower bound** — their true requirement was not pinned down
because they were still saturated even at 1.50\u00d7.

In [ ]:
targets = req[req['req_factor'] > 1.0].sort_values('req_factor', ascending=False).copy()

# nominal MW recovered the same way effective_lrf() does in results_plots_2035.ipynb:
# limit_mw at LRF_REINFORCED / LRF_REINFORCED = nominal (works for AC lines; transformers/DC differ)
lim_by_branch = rein_branch.groupby('branch_name')['limit_mw'].first()
targets['nominal_mw'] = targets['branch_name'].map(lim_by_branch) / LRF_REINFORCED
targets['peak_mw_at_1.5x'] = targets['nominal_mw'] * targets['peak_pct'] / 100.0
targets['status'] = np.where(targets['capped'], 'LOWER BOUND (still capped at 1.50\u00d7)', 'resolved')

report_table = targets[['branch_name', 'nominal_mw', 'peak_mw_at_1.5x', 'req_factor', 'status']] \
    .rename(columns={'branch_name': 'line', 'nominal_mw': 'nominal (MW)',
                     'peak_mw_at_1.5x': 'peak observed (MW)', 'req_factor': 'min factor needed'}) \
    .reset_index(drop=True)
report_table['min factor needed'] = report_table['min factor needed'].round(3)
report_table['nominal (MW)'] = report_table['nominal (MW)'].round(1)
report_table['peak observed (MW)'] = report_table['peak observed (MW)'].round(1)

out_csv = OUT / 'trinity_reinforcement_list.csv'
report_table.to_csv(out_csv, index=False)
print(f'saved: {out_csv}  ({len(report_table)} lines)')
report_table


## Conclusions

1. **Market layer is unaffected.** These `line_rating_factor` sweeps only touch Stage 6 (redispatch);
   DA \u2192 ID2 \u2192 ID3 \u2192 CID \u2192 Balancing clear identically regardless (`[redispatch].from_saved="BAL"`
   reused the same market schedule across every run in this study).
2. **The 2024 grid at its normal operating derate (0.80\u00d7) cannot host the Trinity 2035 AC
   redispatch at all** \u2014 0/48 hours solve.
3. **The need is targeted, not systemic**: of 2 348 branches, only **17** (well under 1%) require any
   reinforcement; the rest of the network is untouched by this fix.
4. **15 of the 17 lines have a precise, validated requirement**, between 1.00\u00d7 and 1.40\u00d7
   nominal. The remaining **2 lines** (`LTGES1146`, `LTGES0668`) are only bounded (\u2265 1.50\u00d7)
   \u2014 pinning their exact value needs a longer/higher cluster probe not run for this report; a
   practical design margin (\u2248 2\u00d7) is a reasonable stand-in.
5. **A blanket 1.50\u00d7 (a superset of the targeted fix) is fully validated**: 336/336 hours solve
   over the two-week cluster sample, against 0/48 at the 0.80\u00d7 baseline. A genuinely *targeted*
   run (only these 17 lines reinforced, the rest left at 0.80\u00d7) was not separately executed, but
   since it is a subset of what the validated 1.50\u00d7 run already reinforces, the same result is
   expected \u2014 flagged here as *inferred*, not directly validated, for the report's methods section.
